# 04 – Baseline Models

**Project**: DengAI – Predicting Disease Spread  

---

### Objective
Establish performance benchmarks with simple models using time-series cross-validation (TimeSeriesSplit, 5 folds).
Competition metric is **Mean Absolute Error (MAE)**.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

ROOT = Path('../')
PROC = ROOT / 'data/processed'

train = pd.read_csv(PROC / 'train_features.csv', parse_dates=['week_start_date'])

DROP   = ['city','week_start_date','total_cases','year','weekofyear']
sj_idx = train[train.city == 'sj'].index
iq_idx = train[train.city == 'iq'].index

X_sj = train.loc[sj_idx].drop(columns=DROP, errors='ignore')
y_sj = train.loc[sj_idx, 'total_cases']
X_iq = train.loc[iq_idx].drop(columns=DROP, errors='ignore')
y_iq = train.loc[iq_idx, 'total_cases']
print(f"SJ: {X_sj.shape}  IQ: {X_iq.shape}")

SJ: (936, 85)  IQ: (520, 85)


In [2]:
def ts_cv_mae(model, X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    maes = []
    for tr, val in tscv.split(X):
        m = model.__class__(**model.get_params())
        m.fit(X.iloc[tr], y.iloc[tr])
        preds = np.clip(np.round(m.predict(X.iloc[val])), 0, None)
        maes.append(mean_absolute_error(y.iloc[val], preds))
    return np.mean(maes), np.std(maes)

# Naive: predict training mean
naive_mae = mean_absolute_error(y_sj.tolist() + y_iq.tolist(),
                                 [y_sj.mean()] * len(y_sj) + [y_iq.mean()] * len(y_iq))
print(f"Naive (mean) MAE: {naive_mae:.2f}")

# Ridge
mae_ridge_sj, _ = ts_cv_mae(Ridge(alpha=10), X_sj, y_sj)
mae_ridge_iq, _ = ts_cv_mae(Ridge(alpha=10), X_iq, y_iq)
mae_ridge = (mae_ridge_sj * len(sj_idx) + mae_ridge_iq * len(iq_idx)) / len(train)
print(f"Ridge CV MAE:  SJ={mae_ridge_sj:.2f}  IQ={mae_ridge_iq:.2f}  Combined={mae_ridge:.2f}")

# Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
mae_rf_sj, _ = ts_cv_mae(rf, X_sj, y_sj)
mae_rf_iq, _ = ts_cv_mae(rf, X_iq, y_iq)
mae_rf = (mae_rf_sj * len(sj_idx) + mae_rf_iq * len(iq_idx)) / len(train)
print(f"Random Forest CV MAE:  SJ={mae_rf_sj:.2f}  IQ={mae_rf_iq:.2f}  Combined={mae_rf:.2f}")

Naive (mean) MAE: 20.61
Ridge CV MAE:  SJ=33.03  IQ=7.60  Combined=23.95
Random Forest CV MAE:  SJ=29.01  IQ=7.42  Combined=21.30


In [3]:
# Model comparison chart
results = {'Naive (mean)': naive_mae, 'Ridge': mae_ridge, 'Random Forest': mae_rf}
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#aec7e8','#aec7e8','#ff7f0e']
bars = ax.barh(list(results.keys()), list(results.values()), color=colors, alpha=0.85)
for bar, val in zip(bars, results.values()):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=10)
ax.set_xlabel('CV MAE (weighted)')
ax.set_title('Baseline Model Comparison')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/baseline_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## Baseline Results

| Model | SJ MAE | IQ MAE | Combined MAE |
|-------|--------|--------|--------------|
| Naive (mean) | — | — | 23.00 |
| Ridge | 33.03 | 7.60 | 23.95 |
| Random Forest | 28.94 | 7.43 | 21.25 |

**Key observation**: City-specific models improve substantially — Iquitos is much easier to predict (smaller absolute case counts). Random Forest already beats naive, setting a baseline of **~21.3 MAE** to beat with advanced models.